# Лабораторная работа № 2. Табличные данные

**Тема:** «Разработка программного продукта бота для определения сорта стекла»

**Исходные данные:** Kaggle — [uciml/glass](https://www.kaggle.com/uciml/glass) (Glass Identification Data Set: 214 образцов стекла, 9 признаков — показатель преломления RI и содержание оксидов Na, Mg, Al, Si, K, Ca, Ba, Fe; целевая переменная `Type` — сорт стекла).

## Цель работы
Освоить работу с табличными данными в `pandas`: первичный анализ, фильтрация, сортировка, группировка, объединение таблиц.

## 1. Теоретическая часть

**1. Что называют табличными данными?** Данные, организованные в виде таблицы: строки — объекты (наблюдения), столбцы — признаки (переменные). В нашей задаче строка — образец стекла, столбцы — показатель преломления и содержание оксидов.

**2. Отличия от других типов данных.**
- *Изображения* — массивы пикселей с пространственной структурой, соседние значения связаны; в таблице столбцы независимы по смыслу и порядок столбцов не важен.
- *Текст* — последовательности переменной длины, требуют токенизации и векторизации; таблица имеет фиксированную схему.
- *Временные ряды* — порядок строк существенен (значение зависит от предыдущих); в таблице строки, как правило, равноправны и независимы.

**3. Почему промышленные ML-задачи в основном работают с табличными данными?** Большая часть корпоративных данных хранится в БД и учётных системах (транзакции, клиенты, датчики, лабораторные анализы), то есть изначально в таблицах; такие данные компактны, интерпретируемы, для них хорошо работают быстрые и объяснимые модели (линейные, деревья, ансамбли).

**4. Типы признаков:** числовые (непрерывные и дискретные), категориальные (номинальные и порядковые), бинарные, а также даты/время и текстовые поля.

**5. Признак и целевая переменная.** Признак — входная характеристика объекта, по которой делается предсказание. Целевая переменная — то, что нужно предсказать. Здесь признаки: RI, Na, Mg, Al, Si, K, Ca, Ba, Fe; целевая переменная: `Type` (сорт стекла).

**6. Различия:** числовые принимают значения на числовой шкале, над ними определены арифметика и порядок; категориальные — конечный набор категорий без арифметического смысла; бинарные — частный случай категориальных с двумя значениями (0/1, да/нет).

**7. Почему корректное кодирование критично?** Модели оперируют числами. Если закодировать категории неправильно (например, целыми числами для непорядковой переменной), модель увидит несуществующий порядок и расстояния; пропуски и «мусорные» значения искажают обучение. Ошибки на этом этапе напрямую ухудшают качество и интерпретируемость модели.

## 2. Практическая часть

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")
#константты
URL  = "https://archive.ics.uci.edu/ml/machine-learning-databases/glass/glass.data"
COLS = ["Id", "RI", "Na", "Mg", "Al", "Si", "K", "Ca", "Ba", "Fe", "Type"]
FEATURES = ["RI", "Na", "Mg", "Al", "Si", "K", "Ca", "Ba", "Fe"]
# Словарь для "перевода" числовых кодов сортов в понятные названия
TYPE_NAMES = {1: "окна зданий (float)", 2: "окна зданий (non-float)",
              3: "окна авто (float)", 5: "контейнеры",
              6: "посуда", 7: "фары"}

def load_glass():
    """Набор Kaggle uciml/glass (glass.csv). Если файл не загружен в Colab,
    берём тот же набор из репозитория UCI."""
    # Проверяем, есть ли файл локально. Если да — читаем его.
    if os.path.exists("glass.csv"):
        return pd.read_csv("glass.csv")
    return pd.read_csv(URL, names=COLS).drop(columns="Id")

df = load_glass()

### Загрузка и первичный анализ

In [ ]:
print("Первые 5 строк:")
display(df.head())
print("Размерность (строк, столбцов):", df.shape)
print("\nТипы данных:")
display(df.dtypes.to_frame("dtype"))
num_cols = df.select_dtypes("number").columns.tolist()
cat_cols = df.select_dtypes(exclude="number").columns.tolist()
print("Числовые столбцы:", num_cols)
print("Категориальные (object) столбцы:", cat_cols or "нет")
print("Целевая переменная Type хранится числом, но по смыслу категориальная (сорт стекла, коды 1,2,3,5,6,7).")

Первые 5 строк:


,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


Размерность (строк, столбцов): (214, 10)

Типы данных:


,dtype
RI,float64
Na,float64
Mg,float64
Al,float64
Si,float64
K,float64
Ca,float64
Ba,float64
Fe,float64
Type,int64


Числовые столбцы: ['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type']
Категориальные (object) столбцы: нет
Целевая переменная Type хранится числом, но по смыслу категориальная (сорт стекла, коды 1,2,3,5,6,7).
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


### 2.1. Базовый уровень

In [ ]:
# Берем только признаки, вызываем describe() (min, max, mean, std и др.)
stats = df[FEATURES].describe().T[["min", "max", "mean", "std"]]
# Создаем новый столбец: Коэффициент вариации = (стандартное отклонение / среднее) * 100%
# Это нужно, чтобы сравнивать разброс признаков с разными масштабами (например, RI ~ 1.5, а Na ~ 13)
stats["разброс, %"] = (stats["std"] / stats["mean"] * 100).round(1)
display(stats.round(3))
# Находим 3 признака с самым большим и самым маленьким относительным разбросом
print("Наиболее вариативные (по коэффициенту вариации):", stats["разброс, %"].nlargest(3).index.tolist())
print("Наиболее стабильные:", stats["разброс, %"].nsmallest(3).index.tolist())

,min,max,mean,std,"разброс, %"
RI,1.511,1.534,1.518,0.003,0.2
Na,10.730,17.380,13.408,0.817,6.1
Mg,0.000,4.490,2.685,1.442,53.7
Al,0.290,3.500,1.445,0.499,34.6
Si,69.810,75.410,72.651,0.775,1.1
K,0.000,6.210,0.497,0.652,131.2
Ca,5.430,16.190,8.957,1.423,15.9
Ba,0.000,3.150,0.175,0.497,284.0
Fe,0.000,0.510,0.057,0.097,170.9


Наиболее вариативные (по коэффициенту вариации): ['Ba', 'Fe', 'K']
Наиболее стабильные: ['RI', 'Si', 'Na']


**Ответ.** Относительный разброс (std/mean) максимален у признаков Ba, Fe, K и Mg — в них много нулей и редкие большие значения (это «маркерные» оксиды для отдельных сортов). Наиболее стабильны Si и RI: основа стекла — кремнезём (≈ 72–73 %), показатель преломления почти не меняется (1.51–1.53). Абсолютное стандартное отклонение больше всего у Mg (1.44) и Ca (1.42).

In [ ]:
# Создаем новую таблицу flt, оставляя только строки, где Mg > 3.5 (булева маска)
flt = df[df["Mg"] > 3.5]
print("Образцов с Mg > 3.5:", len(flt))
print("По возрастанию Mg:")
display(flt.sort_values("Mg").head())
print("По убыванию Mg:")
display(flt.sort_values("Mg", ascending=False).head())

Образцов с Mg > 3.5: 96
По возрастанию Mg:


,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
142,1.51662,12.85,3.51,1.44,73.01,0.68,8.23,0.06,0.25,2
28,1.51768,12.56,3.52,1.43,73.15,0.57,8.54,0.00,0.00,1
78,1.51613,13.92,3.52,1.25,72.88,0.37,7.94,0.00,0.14,2
80,1.51592,12.86,3.52,2.12,72.66,0.69,7.97,0.00,0.00,2
79,1.51590,12.82,3.52,1.90,72.86,0.69,7.97,0.00,0.00,2


По убыванию Mg:


,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.00,1
132,1.51813,13.43,3.98,1.18,72.49,0.58,8.15,0.0,0.00,2
114,1.51847,13.10,3.97,1.19,72.44,0.60,8.43,0.0,0.00,2
133,1.51800,13.71,3.93,1.54,71.81,0.54,8.21,0.0,0.15,2
116,1.51829,13.24,3.90,1.41,72.33,0.55,8.31,0.0,0.10,2


### 2.2. Группировка и агрегация

In [ ]:
# Создаем новый столбец "Сорт", заменяя числа из "Type" на строки из словаря TYPE_NAME
df["Сорт"] = df["Type"].map(TYPE_NAMES)
g = df.groupby("Сорт")["Al"].agg(["mean", "min", "max", "count"]).round(3)
# Сортируем результат по количеству образцов (count) по убыванию
display(g.sort_values("count", ascending=False))
# idxmax() возвращает имя индекса (название сорта), где значение максимально
print("Самая многочисленная группа:", g["count"].idxmax(), "-", g["count"].max())
print("Наибольшее среднее Al:", g["mean"].idxmax())

,mean,min,max,count
Сорт,,,,
окна зданий (non-float),1.408,0.56,2.12,76
окна зданий (float),1.164,0.29,1.69,70
фары,2.123,1.19,2.88,29
окна авто (float),1.201,0.58,1.76,17
контейнеры,2.034,1.40,3.50,13
посуда,1.367,0.34,2.09,9


Самая многочисленная группа: окна зданий (non-float) - 76
Наибольшее среднее Al: фары


In [ ]:
df["Ba>0"] = np.where(df["Ba"] > 0, "Ba есть", "Ba нет")
g2 = df.groupby(["Сорт", "Ba>0"]).agg(n=("RI", "size"), Al_mean=("Al", "mean"), Mg_mean=("Mg", "mean"), Na_mean=("Na", "mean")).round(3)
display(g2)
print("Самая частая комбинация:", g2["n"].idxmax(), g2["n"].max())
print("Нетипичные (редкие) комбинации, n <= 3:")
display(g2[g2["n"] <= 3])

n  Al_mean  Mg_mean  Na_mean
Сорт                    Ba>0                                  
контейнеры              Ba есть   2    2.835    2.145   13.430
                        Ba нет   11    1.888    0.525   12.718
окна авто (float)       Ba есть   1    0.750    3.540   13.640
                        Ba нет   16    1.229    3.544   13.424
окна зданий (float)     Ba есть   3    1.290    3.530   13.517
                        Ba нет   67    1.158    3.553   13.230
окна зданий (non-float) Ba есть   6    1.525    2.507   12.673
                        Ba нет   70    1.398    3.045   13.149
посуда                  Ba нет    9    1.367    1.306   14.647
фары                    Ba есть  26    2.195    0.387   14.560
                        Ba нет    3    1.493    1.847   13.417

Самая частая комбинация: ('окна зданий (non-float)', 'Ba нет') 70
Нетипичные (редкие) комбинации, n <= 3:


,,n,Al_mean,Mg_mean,Na_mean
Сорт,Ba>0,,,,
контейнеры,Ba есть,2,2.835,2.145,13.430
окна авто (float),Ba есть,1,0.750,3.540,13.640
окна зданий (float),Ba есть,3,1.290,3.530,13.517
фары,Ba нет,3,1.493,1.847,13.417


**Выводы по группировке.** Самые многочисленные сорта — окна зданий non-float (76 образцов) и float (70); «посуда» (9) и «контейнеры» (13) малочисленны. Наибольшее среднее значение Al — у «фар» (≈ 2.12), у «контейнеров» оно также высокое (≈ 2.03), у окон — 1.2–1.4. Группы различаются по составу: у окон (зданий и авто) Mg ≈ 3–3.5 %, у «контейнеров» и «фар» магния мало (0.5–0.8 %); «контейнеры» богаты K и Ca; «посуда» — натрием (≈ 14.7 %) и без калия. Барий практически присутствует только в «фарах» (26 из 29 образцов), поэтому образцы с барием вне «фар» (например, 6 образцов окон non-float) и «фары» без бария (3 образца) можно считать нетипичными.

### 2.3. Объединение таблиц

In [ ]:
# справочник сортов (ключ Type) и описание применения
ref = pd.DataFrame({"Type": [1, 2, 3, 5, 6, 7, 4],
                    "Название": ["Building windows float", "Building windows non-float", "Vehicle windows float",
                                 "Containers", "Tableware", "Headlamps", "Vehicle windows non-float (нет в данных)"],
                    "Область": ["строительство", "строительство", "транспорт", "упаковка", "бытовая", "транспорт", "транспорт"]})
merged = df[["RI", "Na", "Mg", "Al", "Type"]].merge(ref, on="Type", how="left")        # merge по ключу Type
print("merge: размер", merged.shape, "| пропусков:", int(merged.isna().sum().sum()))
display(merged.sample(5, random_state=1))

merged_outer = df[["Type"]].drop_duplicates().merge(ref, on="Type", how="outer")        # outer: видна запись без образцов
print("outer merge: размер", merged_outer.shape)
display(merged_outer.sort_values("Type"))

# concat: разбиваем таблицу на две части и склеиваем обратно
parts = [df.iloc[:100], df.iloc[100:]]
back = pd.concat(parts, ignore_index=True)
print("concat: размер", back.shape, "| совпадает с исходной:", back.equals(df))
# concat по столбцам с признаком, у которого индекс не совпадает -> пропуски
extra = pd.DataFrame({"Ratio_Na_Ca": (df["Na"] / df["Ca"]).iloc[:150]})
wide = pd.concat([df[["Na", "Ca"]], extra], axis=1)
print("concat axis=1 с неполным столбцом: пропусков =", int(wide.isna().sum().sum()), "(у строк 150-213 нет значения)")

merge: размер (214, 7) | пропусков: 0


,RI,Na,Mg,Al,Type,Название,Область
108,1.52222,14.43,0.00,1.00,2,Building windows non-float,строительство
206,1.51645,14.94,0.00,1.87,7,Headlamps,транспорт
106,1.53125,10.73,0.00,2.10,2,Building windows non-float,строительство
107,1.53393,12.30,0.00,1.00,2,Building windows non-float,строительство
51,1.51926,13.20,3.33,1.28,1,Building windows float,строительство


outer merge: размер (7, 3)


,Type,Название,Область
0,1,Building windows float,строительство
1,2,Building windows non-float,строительство
2,3,Vehicle windows float,транспорт
3,4,Vehicle windows non-float (нет в данных),транспорт
4,5,Containers,упаковка
5,6,Tableware,бытовая
6,7,Headlamps,транспорт


concat: размер (214, 12) | совпадает с исходной: True
concat axis=1 с неполным столбцом: пропусков = 64 (у строк 150-213 нет значения)


**Ответы.** Объединение выполнялось по ключу `Type`. К образцам добавились название сорта и область применения. В `merge(how="left")` пропусков нет, потому что все коды в данных есть в справочнике; в `outer` появилась строка сорта 4, для которой в данных нет образцов (остальные поля — NaN). При `concat(axis=1)` с неполным столбцом пропуски возникли, так как строки с индексами 150–213 не имеют значений нового признака.

### вывод

Исходные данные представляют собой таблицу из 214 образцов стекла и 10 числовых столбцов: показатель преломления RI, содержание оксидов Na, Mg, Al, Si, K, Ca, Ba, Fe (в весовых %) и целевой код сорта Type. Категориальных столбцов типа `object` нет — сорт закодирован числом, поэтому по смыслу переменная `Type` категориальная. Пропусков в таблице нет, один образец является полным дубликатом другого (будет удалён в ЛР 5). Наиболее полезными операциями pandas оказались `describe()`, `groupby().agg()`, булевая фильтрация и `merge()` — вместе они позволяют быстро получить портрет данных. Распределение классов несбалансированное: сорта 1 и 2 составляют большую часть выборки, сорта 5, 6 малочисленны. Признаки Ba, Fe, K и Mg имеют большой относительный разброс и много нулевых значений, а Si и RI — самые стабильные. Группы заметно различаются по составу: фары выделяются барием (в среднем 1.04 %), контейнеры — калием (1.47 %), кальцием и малым содержанием магния, посуда — натрием и отсутствием калия, окна — высоким магнием. Это означает, что состав оксидов действительно несёт информацию о сорте и задача классификации разрешима. Объединение с справочником сортов прошло без потерь, пропуски возникали только там, где ключ отсутствовал.